<a href="https://colab.research.google.com/github/RafihaikalP/BigData26_A_2411532002_Rafi-Haikal-Pratama/blob/main/Praktikum2/BD_A_P02_2411532002_Rafi_Haikal_Pratama_Latihan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random
import os
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

path_drive = "/content/drive/My Drive/BigData/Praktikum2/"
os.makedirs(path_drive, exist_ok=True)

print("Google Drive berhasil terhubung!")
print(f"File akan disimpan di: {path_drive}")

Mounted at /content/drive
Google Drive berhasil terhubung!
File akan disimpan di: /content/drive/My Drive/BigData/Praktikum2/


# **Fungsi Pipeline**

In [ ]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

Kedua fungsi ini diletakkan di awal agar bisa dipanggil berkali-kali oleh setiap latihan tanpa perlu menulis ulang. bersihkan_harga() membersihkan kolom price dari berbagai format teks menjadi angka float. parse_tanggal() mencoba tiga format tanggal secara berurutan sampai salah satu berhasil.

# **Buat dan Bersihkan Dataset**

In [ ]:
def buat_dataset(seed):
    # Inisialisasi
    np.random.seed(seed)
    random.seed(seed)
    fake = Faker("id_ID")
    Faker.seed(seed)

    N = 500
    kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
    metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

    # Buat 500 baris transaksi
    rows = []
    for i in range(1, N + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fake.name()
        produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
        qty = random.randint(1, 5)

        harga_variants = [
            str(harga_dasar),
            f"Rp{harga_dasar:,}".replace(",", "."),
            f"{harga_dasar}.0",
            f" {harga_dasar} ",
        ]
        harga = random.choice(harga_variants)

        tgl = fake.date_between(start_date="-90d", end_date="today")
        tgl_variants = [
            tgl.strftime("%Y-%m-%d"),
            tgl.strftime("%d/%m/%Y"),
            tgl.strftime("%d-%m-%Y")
        ]
        tanggal = random.choice(tgl_variants)

        metode = random.choice(metode_bayar)
        if random.random() < 0.3:
            metode = metode.lower()
        if random.random() < 0.2:
            kategori = kategori.upper() + " "

        kota = fake.city()
        rating = random.choice([1, 2, 3, 4, 5, None, None])

        rows.append({
            "transaction_id": trx_id,
            "customer_name": nama_pelanggan,
            "product_name": produk.strip(),
            "category": kategori,
            "price": harga,
            "quantity": qty,
            "payment_method": metode,
            "transaction_date": tanggal,
            "shipping_city": kota,
            "rating": rating,
        })

    df = pd.DataFrame(rows)

    # Sisipkan missing value
    for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        idx = df.sample(frac=frac, random_state=seed).index
        df.loc[idx, col] = np.nan

    # Sisipkan duplikat
    dup_rows = df.sample(n=15, random_state=seed)
    df = pd.concat([df, dup_rows], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    jumlah_mentah = len(df)

    # Preprocessing
    df = df.dropna(subset=["customer_name", "payment_method"])
    df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
    df = df.drop_duplicates()

    for col in ["category", "payment_method", "shipping_city"]:
        df[col] = df[col].astype("string").str.strip().str.title()
    df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

    df["price"] = df["price"].apply(bersihkan_harga).astype(float)
    df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
    df["quantity"] = df["quantity"].astype(int)

    jumlah_bersih = len(df)

    return df, jumlah_mentah, jumlah_bersih

Seluruh pipeline dari K-2 sampai K-5 dibungkus dalam satu fungsi buat_dataset(seed) yang menerima nilai seed sebagai parameter. Tujuannya agar latihan 1 bisa menjalankan pipeline yang sama dengan seed berbeda hanya dengan satu baris pemanggilan, tanpa perlu menulis ulang ratusan baris kode. Fungsi ini mengembalikan tiga nilai: DataFrame yang sudah bersih, jumlah baris data mentah, dan jumlah baris data bersih.

# **Latihan 1 — Jalankan Pipeline dengan SEED = 7**

In [ ]:

print("LATIHAN 1 Perbandingan SEED = 42 vs SEED = 7")

# Jalankan pipeline dengan kedua seed
df_42, mentah_42, bersih_42 = buat_dataset(seed=42)
df_7,  mentah_7,  bersih_7  = buat_dataset(seed=7)

# Tampilkan perbandingan
print(f"\n{'Keterangan':<25} {'SEED = 42':>10} {'SEED = 7':>10}")

print(f"{'Baris data mentah':<25} {mentah_42:>10} {mentah_7:>10}")
print(f"{'Baris data bersih':<25} {bersih_42:>10} {bersih_7:>10}")
print(f"{'Baris terbuang':<25} {mentah_42 - bersih_42:>10} {mentah_7 - bersih_7:>10}")


print("\nKesimpulan:")
if bersih_42 != bersih_7:
    print(f"Jumlah baris BERBEDA ({bersih_42} vs {bersih_7}).")
else:
    print(f"Jumlah baris SAMA ({bersih_42} baris).")

print("Ini karena nilai SEED mengendalikan posisi missing value")
print("dan baris duplikat yang disisipkan secara acak.")
print("SEED berbeda → distribusi masalah berbeda → hasil akhir berbeda.")

LATIHAN 1 — Perbandingan SEED = 42 vs SEED = 7

Keterangan                 SEED = 42   SEED = 7
Baris data mentah                515        515
Baris data bersih                490        490
Baris terbuang                    25         25

Kesimpulan:
Jumlah baris SAMA (490 baris).
Ini karena nilai SEED mengendalikan posisi missing value
dan baris duplikat yang disisipkan secara acak.
SEED berbeda → distribusi masalah berbeda → hasil akhir berbeda.


Latihan 1 membuktikan bahwa hasil pipeline sepenuhnya dikendalikan oleh nilai SEED. Dengan memanggil fungsi buat_dataset() dua kali menggunakan seed yang berbeda, kita bisa langsung membandingkan hasilnya secara berdampingan. Jika jumlah baris akhir berbeda, itu karena SEED = 7 menghasilkan distribusi missing value dan posisi duplikat yang berbeda dari SEED = 42 — bukan karena ada yang salah dalam kodenya.

# **Latihan 2 — Kolom is_valid_price**

In [ ]:
print("LATIHAN 2 Validasi Kolom Price")

# Gunakan hasil dari SEED = 42
df = df_42.copy()

# Tambahkan kolom validasi
df["is_valid_price"] = df["price"] > 0

# Ringkasan hasil
print("\nDistribusi is_valid_price:")
print(df["is_valid_price"].value_counts())

jumlah_valid     = (df["is_valid_price"] == True).sum()
jumlah_not_valid = (df["is_valid_price"] == False).sum()
total            = len(df)

print(f"\nTotal baris       : {total}")
print(f"Harga valid (> 0) : {jumlah_valid} baris ({jumlah_valid/total*100:.1f}%)")
print(f"Harga tidak valid : {jumlah_not_valid} baris ({jumlah_not_valid/total*100:.1f}%)")

# Tampilkan baris tidak valid jika ada
if jumlah_not_valid > 0:
    print("\nDetail baris dengan harga tidak valid:")
    print(df[df["is_valid_price"] == False][["transaction_id", "price", "is_valid_price"]])
else:
    print("\nSemua harga valid tidak ada harga nol atau negatif.")

LATIHAN 2 Validasi Kolom Price

Distribusi is_valid_price:
is_valid_price
True    490
Name: count, dtype: int64

Total baris       : 490
Harga valid (> 0) : 490 baris (100.0%)
Harga tidak valid : 0 baris (0.0%)

Semua harga valid — tidak ada harga nol atau negatif.


Latihan 2 menambahkan kolom baru is_valid_price yang berisi True atau False berdasarkan kondisi df["price"] > 0. Kolom seperti ini disebut kolom flag atau kolom validasi — fungsinya sebagai penanda kualitas data yang bisa langsung dipakai untuk memfilter baris bermasalah tanpa harus mengubah data aslinya. Selain itu ditampilkan juga ringkasan persentase harga valid dan tidak valid agar mudah dibaca.

# **Latihan 3 — Jumlah Transaksi per Kategori**

In [ ]:

print("LATIHAN 3 Transaksi per Kategori")

# Gunakan hasil dari SEED = 42
df = df_42.copy()

# Hitung transaksi per kategori
per_kategori = df["category"].value_counts()
total = per_kategori.sum()

print("\nJumlah transaksi per kategori:")

for kategori, jumlah in per_kategori.items():
    persen = jumlah / total * 100
    bar    = "█" * int(persen / 2)
    print(f"{kategori:<15} {jumlah:>4} transaksi  {persen:>5.1f}%  {bar}")


print(f"{'TOTAL':<15} {total:>4} transaksi  100.0%")

# Verifikasi
print(f"\nTotal dari value_counts() : {total}")
print(f"Total baris df            : {len(df)}")
print(f"Cocok                     : {total == len(df)}")

LATIHAN 3 Transaksi per Kategori

Jumlah transaksi per kategori:
Olahraga          97 transaksi   19.8%  █████████
Kesehatan         91 transaksi   18.6%  █████████
Elektronik        89 transaksi   18.2%  █████████
Buku              82 transaksi   16.7%  ████████
Fashion           66 transaksi   13.5%  ██████
Rumah Tangga      65 transaksi   13.3%  ██████
TOTAL            490 transaksi  100.0%

Total dari value_counts() : 490
Total baris df            : 490
Cocok                     : True


Latihan 3 menggunakan value_counts() untuk menghitung berapa transaksi yang ada di setiap kategori produk. Selain menampilkan angkanya, kode ini juga menambahkan persentase dan visualisasi bar sederhana menggunakan karakter agar distribusi antar kategori lebih mudah dibaca secara sekilas. Di bagian akhir ada verifikasi bahwa total dari value_counts() sama dengan jumlah baris DataFrame jika tidak sama berarti ada baris yang kolom kategorinya kosong.

In [ ]:
# Simpan hasil SEED = 42
df_42.to_csv(path_drive + "latihan_bersih_seed42.csv", index=False)
df_42.to_csv("latihan_bersih_seed42.csv", index=False)

# Simpan hasil SEED = 7
df_7.to_csv(path_drive + "latihan_bersih_seed7.csv", index=False)
df_7.to_csv("latihan_bersih_seed7.csv", index=False)

print(f"\nlatihan_bersih_seed42.csv → {bersih_42} baris")
print(f"latihan_bersih_seed7.csv  → {bersih_7} baris")


MENYIMPAN HASIL KE GOOGLE DRIVE

latihan_bersih_seed42.csv → 490 baris
latihan_bersih_seed7.csv  → 490 baris

Semua file tersimpan di:
/content/drive/My Drive/BigData/Praktikum2/
